# CytoSignal diffusion LR scores for slides 1–9

This notebook processes `slide1_clean_manual_EBs.h5ad` through `slide9_clean_manual_EBs.h5ad`. Within each file, cells are separated by `obs['EB_cluster_manual']`. It calculates only `diffusion-Raw_smooth` at 100, 200, and 400 µm, without permutation or significance testing.

The original H5AD files are read-only. Results are written to a separate output directory.

In [32]:
options(width = 140)

library(cytosignal)
library(hdf5r)
library(Matrix)

cat("R:", R.version.string, "\n")
cat("cytosignal:", as.character(packageVersion("cytosignal")), "\n")
cat("hdf5r:", as.character(packageVersion("hdf5r")), "\n")

R: R version 4.3.3 (2024-02-29) 
cytosignal: 0.5.1 


hdf5r: 1.3.12 


In [33]:
INPUT_DIR <- "/Users/newuser/test"
SLIDE_FILES <- file.path(
  INPUT_DIR,
  sprintf("slide%d_clean_manual_EBs.h5ad", 1:9)
)
names(SLIDE_FILES) <- paste0("slide", 1:9)

OUTPUT_DIR <- file.path(INPUT_DIR, "cytosignal_9slides_diffusion_scores_only")
EB_KEY <- "EB_cluster_manual"
RADII_UM <- c(100, 200, 400)
SCALE_FACTOR <- 1

# Josh Welch recommended a very small jitter for exact duplicate coordinates.
# The original H5AD coordinates are never changed; jitter is used only in
# memory for Delaunay construction.
DUPLICATE_COORDINATE_POLICY <- "jitter_for_dt"
stopifnot(DUPLICATE_COORDINATE_POLICY %in% c("stop", "jitter_for_dt"))

missing_files <- SLIDE_FILES[!file.exists(SLIDE_FILES)]
if (length(missing_files)) stop("Missing input files: ", paste(missing_files, collapse = ", "))
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

cat("Input files:", length(SLIDE_FILES), "\n")
cat("Radii:", paste(RADII_UM, collapse = ", "), "µm\n")
cat("Duplicate-coordinate policy:", DUPLICATE_COORDINATE_POLICY, "\n")

Input files: 9 
Radii: 100, 200, 400 µm
Duplicate-coordinate policy: jitter_for_dt 


## H5AD categorical reader and preflight

The preflight reads only cell IDs, `EB_cluster_manual`, and `X_spatial`; it does not calculate scores.

In [34]:
read_h5ad_categorical <- function(filename, key) {
  h5 <- H5File$new(filename, mode = "r")
  on.exit(h5$close_all())
  node <- h5[[paste0("obs/", key)]]
  if (!inherits(node, "H5Group") ||
      !all(c("categories", "codes") %in% names(node))) {
    stop("obs['", key, "'] is not stored as an H5AD categorical column in ", filename)
  }
  categories <- as.character(node[["categories"]][])
  codes <- as.integer(node[["codes"]][])
  values <- rep(NA_character_, length(codes))
  valid <- codes >= 0L
  values[valid] <- categories[codes[valid] + 1L]
  values
}

read_h5ad_obs_names <- function(filename) {
  h5 <- H5File$new(filename, mode = "r")
  on.exit(h5$close_all())
  index_key <- h5[["obs"]]$attr_open("_index")$read()
  as.character(h5[[paste0("obs/", index_key)]][])
}

read_h5ad_spatial <- function(filename) {
  h5 <- H5File$new(filename, mode = "r")
  on.exit(h5$close_all())
  # hdf5r exposes this H5AD dataset as 2 x cells; transpose to cells x 2.
  t(as.matrix(h5[["obsm/X_spatial"]][,]))
}

H5ADToCS_compatible <- function(
  filename,
  layer = "layers/counts",
  spatialKey = "X_spatial",
  clusterKey = "cell_type"
) {
  h5 <- H5File$new(filename, mode = "r")
  on.exit(h5$close_all())

  mat_obj <- h5[[layer]]
  required <- c("data", "indices", "indptr")
  if (!inherits(mat_obj, "H5Group") || !all(required %in% names(mat_obj))) {
    stop("Selected layer is not a supported sparse H5AD matrix: ", layer)
  }

  x <- mat_obj[["data"]][]
  indices <- as.integer(mat_obj[["indices"]][])
  indptr <- as.integer(mat_obj[["indptr"]][])
  shape <- as.integer(h5attr(mat_obj, "shape"))
  encoding <- as.character(h5attr(mat_obj, "encoding-type"))

  if (encoding == "csr_matrix") {
    # H5AD is cells x genes in CSR. Reinterpret its rows as columns to obtain genes x cells.
    mat <- Matrix::sparseMatrix(
      i = indices + 1L,
      p = indptr,
      x = x,
      dims = rev(shape),
      repr = "C"
    )
  } else if (encoding == "csc_matrix") {
    # H5AD is cells x genes in CSC. Construct it normally, then transpose.
    cells_by_genes <- Matrix::sparseMatrix(
      i = indices + 1L,
      p = indptr,
      x = x,
      dims = shape,
      repr = "C"
    )
    mat <- methods::as(t(cells_by_genes), "dgCMatrix")
  } else {
    stop("Unsupported sparse encoding: ", encoding)
  }

  obs_index_key <- as.character(h5attr(h5[["obs"]], "_index"))
  var_index_key <- as.character(h5attr(h5[["var"]], "_index"))
  obs_names <- as.character(h5[[paste0("obs/", obs_index_key)]][])
  var_names <- as.character(h5[[paste0("var/", var_index_key)]][])
  dimnames(mat) <- list(var_names, obs_names)

  cluster_node <- h5[[paste0("obs/", clusterKey)]]
  if (!inherits(cluster_node, "H5Group") ||
      !all(c("categories", "codes") %in% names(cluster_node))) {
    stop("clusterKey is not a categorical H5AD column: ", clusterKey)
  }
  cluster_categories <- as.character(cluster_node[["categories"]][])
  cluster_codes <- as.integer(cluster_node[["codes"]][])
  cluster_values <- rep(NA_character_, length(cluster_codes))
  valid_cluster <- cluster_codes >= 0L
  cluster_values[valid_cluster] <- cluster_categories[cluster_codes[valid_cluster] + 1L]
  clusters <- factor(cluster_values, levels = cluster_categories)
  names(clusters) <- obs_names

  loc <- t(as.matrix(h5[[paste0("obsm/", spatialKey)]][,]))
  if (!identical(nrow(loc), length(obs_names))) {
    stop("Spatial coordinate rows do not match observation count")
  }
  rownames(loc) <- obs_names
  colnames(loc) <- c("x", "y")

  createCytoSignal(raw.data = mat, cells.loc = loc, clusters = clusters)
}

count_duplicate_rows <- function(x) {
  sum(duplicated(as.data.frame(x[, 1:2, drop = FALSE])))
}

preflight_rows <- list()
for (slide_id in names(SLIDE_FILES)) {
  filename <- SLIDE_FILES[[slide_id]]
  eb <- read_h5ad_categorical(filename, EB_KEY)
  xy <- read_h5ad_spatial(filename)
  eb_ids <- unique(eb[!is.na(eb)])
  eb_ids <- eb_ids[order(as.integer(sub("^EB_", "", eb_ids)))]
  for (eb_id in eb_ids) {
    idx <- which(eb == eb_id)
    preflight_rows[[length(preflight_rows) + 1L]] <- data.frame(
      slide = slide_id,
      EB = eb_id,
      cells = length(idx),
      duplicate_coordinate_rows = count_duplicate_rows(xy[idx, , drop = FALSE]),
      stringsAsFactors = FALSE
    )
  }
}
preflight <- do.call(rbind, preflight_rows)
cat("Total slide-specific EBs:", nrow(preflight), "\n")
cat("EBs with duplicated coordinates:", sum(preflight$duplicate_coordinate_rows > 0), "\n")
print(aggregate(cbind(EBs = cells, duplicate_EBs = duplicate_coordinate_rows) ~ slide,
                transform(preflight,
                          cells = 1L,
                          duplicate_coordinate_rows = as.integer(duplicate_coordinate_rows > 0)),
                sum))
write.csv(preflight, file.path(OUTPUT_DIR, "preflight_EB_coordinate_summary.csv"), row.names = FALSE)

Total slide-specific EBs: 771 
EBs with duplicated coordinates: 199 
   slide EBs duplicate_EBs
1 slide1  98             6
2 slide2 132            48
3 slide3 133            11
4 slide4  99            12
5 slide5  45            13
6 slide6  40            20
7 slide7  34            13
8 slide8 116            64
9 slide9  74            12


## Load one slide and prepare one manual EB

In [35]:
load_slide <- function(slide_id) {
  filename <- SLIDE_FILES[[slide_id]]
  if (is.null(filename)) stop("Unknown slide: ", slide_id)

  cs_full <- H5ADToCS_compatible(
    filename = filename,
    layer = "layers/counts",
    spatialKey = "X_spatial",
    clusterKey = "cell_type"
  )
  eb_manual <- read_h5ad_categorical(filename, EB_KEY)
  names(eb_manual) <- colnames(cs_full@raw.counts)

  stopifnot(length(eb_manual) == ncol(cs_full@raw.counts))
  stopifnot(identical(rownames(cs_full@cells.loc), colnames(cs_full@raw.counts)))

  eb_ids <- unique(eb_manual[!is.na(eb_manual)])
  eb_ids <- eb_ids[order(as.integer(sub("^EB_", "", eb_ids)))]
  list(
    slide_id = slide_id,
    filename = filename,
    cs_full = cs_full,
    eb_manual = eb_manual,
    eb_ids = eb_ids
  )
}

# These H5AD files are already cleaned; zero thresholds preserve every input cell.
prepare_slide_eb <- function(slide_data, eb_id, counts_thresh = 0, gene_thresh = 0) {
  if (!eb_id %in% slide_data$eb_ids) stop("Unknown EB in ", slide_data$slide_id, ": ", eb_id)
  keep <- which(slide_data$eb_manual == eb_id)
  if (length(keep) < 3L) stop(slide_data$slide_id, "/", eb_id, " has fewer than three cells")

  object <- createCytoSignal(
    raw.data = slide_data$cs_full@raw.counts[, keep, drop = FALSE],
    cells.loc = slide_data$cs_full@cells.loc[keep, , drop = FALSE],
    clusters = droplevels(slide_data$cs_full@clusters[keep])
  )

  upper_names <- toupper(rownames(object@raw.counts))
  if (anyDuplicated(upper_names)) {
    stop("Upper-casing gene names creates duplicates in ", slide_data$slide_id, "/", eb_id)
  }
  rownames(object@raw.counts) <- upper_names

  object <- addIntrDB(object, g_to_u, db.diff, db.cont, inter.index)
  object <- removeLowQuality(
    object,
    counts.thresh = counts_thresh,
    gene.thresh = gene_thresh
  )
  object <- changeUniprot(object)
  object
}

## Duplicate-coordinate handling and scores-only calculation

With policy `stop`, duplicated-coordinate EBs are recorded as failures. With policy `jitter_for_dt`, a deterministic 0.0001 µm displacement is used only while constructing the Delaunay graph; original coordinates are restored before Gaussian diffusion.

In [36]:
jitter_exact_duplicates <- function(loc, slide_id, eb_id, amount = 1e-4) {
  xy <- as.data.frame(loc[, 1:2, drop = FALSE])
  # Keep the first cell at each coordinate unchanged and move only later duplicates.
  duplicate_cells <- duplicated(xy)
  n_jittered <- sum(duplicate_cells)
  if (n_jittered == 0L) return(loc)

  slide_number <- as.integer(sub("^slide", "", slide_id))
  eb_number <- as.integer(sub("^EB_", "", eb_id))
  set.seed(20260000L + slide_number * 1000L + eb_number)
  angles <- runif(n_jittered, 0, 2 * pi)
  delta <- amount * cbind(cos(angles), sin(angles))
  out <- loc
  out[duplicate_cells, 1:2] <-
    loc[duplicate_cells, 1:2, drop = FALSE] + delta
  out
}

findNNRaw_with_policy <- function(object, slide_id, eb_id) {
  original_loc <- object@cells.loc
  n_dup <- count_duplicate_rows(original_loc)

  if (n_dup == 0L) {
    return(list(object = findNNRaw(object), duplicate_rows = 0L, dt_jitter_um = 0))
  }

  if (DUPLICATE_COORDINATE_POLICY == "stop") {
    stop(slide_id, "/", eb_id, " has ", n_dup,
         " duplicated coordinate row(s); policy is 'stop'")
  }

  object@cells.loc <- jitter_exact_duplicates(
    original_loc, slide_id = slide_id, eb_id = eb_id, amount = 1e-4
  )
  message(slide_id, "/", eb_id, ": applied 1e-4 µm jitter to ",
          n_dup, " duplicated-coordinate cell(s)")
  object <- findNNRaw(object)
  object@cells.loc <- original_loc
  list(object = object, duplicate_rows = n_dup, dt_jitter_um = 1e-4)
}

write_interaction_metadata <- function(interaction_ids) {
  metadata_file <- file.path(OUTPUT_DIR, "interaction_metadata.csv")
  uniprot_to_gene <- setNames(g_to_u$hgnc_symbol, g_to_u$uniprot)

  side_to_gene <- function(side, interaction_id) {
    uniprot <- names(side)[as.character(side) == interaction_id]
    genes <- unname(uniprot_to_gene[uniprot])
    genes[is.na(genes) | genes == ""] <- uniprot[is.na(genes) | genes == ""]
    paste(unique(genes), collapse = "+")
  }

  new_metadata <- data.frame(
    interaction_id = interaction_ids,
    ligand_name = vapply(interaction_ids, function(id) side_to_gene(db.diff$ligands, id), character(1)),
    receptor_name = vapply(interaction_ids, function(id) side_to_gene(db.diff$receptors, id), character(1)),
    stringsAsFactors = FALSE
  )
  new_metadata$lr_name <- paste(new_metadata$ligand_name, new_metadata$receptor_name, sep = "_")
  new_metadata$lr_type <- "diffusion"

  if (file.exists(metadata_file)) {
    old_metadata <- read.csv(metadata_file, stringsAsFactors = FALSE)
    new_metadata <- rbind(old_metadata, new_metadata)
    new_metadata <- new_metadata[!duplicated(new_metadata$interaction_id), , drop = FALSE]
  }
  write.csv(new_metadata, metadata_file, row.names = FALSE)
  invisible(metadata_file)
}

score_slide_eb_radius <- function(base_object, slide_id, eb_id, radius_um) {
  object <- base_object
  object <- inferEpsParams(
    object,
    scale.factor = SCALE_FACTOR,
    r.eps.real = radius_um
  )

  raw_result <- findNNRaw_with_policy(object, slide_id, eb_id)
  object <- raw_result$object
  object <- findNNGauEB(object, imp.name = "diffusion")
  object <- imputeNiche(object, imp.use = "diffusion", weights = "none")

  object <- inferScoreLR(
    object,
    lig.imp = "diffusion",
    recep.imp = "Raw",
    norm.method = "default",
    intr.db.use = "diff_dep",
    lrscore.name = "diffusion-Raw"
  )

  score_raw <- object@lrscore[["diffusion-Raw"]]@score
  diffusion_graph <- object@imputation[["diffusion"]]@nn.graph
  score_smooth <- Matrix::Matrix(t(t(score_raw) %*% diffusion_graph), sparse = TRUE)
  dimnames(score_smooth) <- dimnames(score_raw)

  result <- list(
    slide = slide_id,
    EB = eb_id,
    radius_um = radius_um,
    sigma_um = object@parameters$sigma.scale,
    score_slot = "diffusion-Raw_smooth",
    duplicate_coordinate_rows = raw_result$duplicate_rows,
    dt_jitter_um = raw_result$dt_jitter_um,
    score = score_smooth
  )

  slide_output <- file.path(OUTPUT_DIR, slide_id)
  dir.create(slide_output, recursive = TRUE, showWarnings = FALSE)
  prefix <- sprintf("%s_%s_diffusion-Raw_smooth_radius%03dum", slide_id, eb_id, radius_um)
  rds_file <- file.path(slide_output, paste0(prefix, ".rds"))
  csv_file <- file.path(slide_output, paste0(prefix, "_nonzero.csv.gz"))
  saveRDS(result, rds_file)

  triplet <- Matrix::summary(score_smooth)
  n_nonzero <- nrow(triplet)
  long <- data.frame(
    slide = rep(slide_id, n_nonzero),
    EB = rep(eb_id, n_nonzero),
    radius_um = rep(radius_um, n_nonzero),
    cell_id = rownames(score_smooth)[triplet$i],
    lr_type = rep("diffusion", n_nonzero),
    score_slot = rep("diffusion-Raw_smooth", n_nonzero),
    interaction_id = colnames(score_smooth)[triplet$j],
    lr_score = triplet$x,
    stringsAsFactors = FALSE
  )
  con <- gzfile(csv_file, open = "wt")
  write.csv(long, con, row.names = FALSE)
  close(con)
  write_interaction_metadata(colnames(score_smooth))

  message(slide_id, "/", eb_id,
          " | radius=", radius_um,
          " µm | sigma=", round(result$sigma_um, 2),
          " µm | matrix=", nrow(score_smooth), " x ", ncol(score_smooth),
          " | duplicate rows=", result$duplicate_coordinate_rows)
  result
}

expected_result_files <- function(slide_id, eb_id, radius_um) {
  prefix <- sprintf(
    "%s_%s_diffusion-Raw_smooth_radius%03dum", slide_id, eb_id, radius_um
  )
  file.path(
    OUTPUT_DIR, slide_id,
    c(paste0(prefix, ".rds"), paste0(prefix, "_nonzero.csv.gz"))
  )
}

run_slide_eb <- function(slide_data, eb_id) {
  radius_complete <- vapply(
    RADII_UM,
    function(radius_um) all(file.exists(expected_result_files(
      slide_data$slide_id, eb_id, radius_um
    ))),
    logical(1)
  )
  if (all(radius_complete)) {
    message("Skipping complete result: ", slide_data$slide_id, "/", eb_id)
    return(invisible(NULL))
  }

  base_object <- prepare_slide_eb(slide_data, eb_id)
  for (radius_um in RADII_UM) {
    outputs <- expected_result_files(slide_data$slide_id, eb_id, radius_um)
    if (all(file.exists(outputs))) {
      message("Skipping complete radius: ", slide_data$slide_id, "/", eb_id,
              " radius=", radius_um, " µm")
      next
    }
    result <- score_slide_eb_radius(base_object, slide_data$slide_id, eb_id, radius_um)
    rm(result)
    gc()
  }
  invisible(NULL)
}

## Optional pilot

`slide1/EB_01` has no duplicated coordinates in the current files. Enabling the pilot switch runs the test at all three radii.

In [37]:
RUN_PILOT <- TRUE

if (RUN_PILOT) {
  pilot_slide <- load_slide("slide1")
  run_slide_eb(pilot_slide, "EB_01")
  rm(pilot_slide)
  gc()
} else {
  message("Pilot not started.")
}

Skipping complete result: slide1/EB_01



,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,2169389,115.9,4037052,215.7,NA,4037052,215.7
Vcells,3866610,29.5,15542488,118.6,65536,194536409,1484.2


## Resumable batch across all nine slides

A radius is complete only when both its `.rds` and `_nonzero.csv.gz` exist. Fully completed results are skipped, and partially written radii are rerun. Exact duplicate coordinates receive a reproducible `1e-4` µm in-memory jitter for Delaunay construction. Remaining failures are written to `batch_failures.csv`.

In [38]:
RUN_ALL_SLIDES <- TRUE

if (RUN_ALL_SLIDES) {
  failures <- list()

  for (slide_id in names(SLIDE_FILES)) {
    message("\n========== ", slide_id, " ==========")
    slide_data <- load_slide(slide_id)

    for (eb_id in slide_data$eb_ids) {
      tryCatch({
        run_slide_eb(slide_data, eb_id)
      }, error = function(e) {
        key <- paste(slide_id, eb_id, sep = "/")
        failures[[key]] <<- conditionMessage(e)
        message("FAILED ", key, ": ", conditionMessage(e))
      })
    }

    rm(slide_data)
    gc()
  }

  if (length(failures)) {
    failure_table <- data.frame(
      slide_EB = names(failures),
      error = unlist(failures, use.names = FALSE),
      stringsAsFactors = FALSE
    )
  } else {
    failure_table <- data.frame(slide_EB = character(), error = character())
  }
  write.csv(failure_table, file.path(OUTPUT_DIR, "batch_failures.csv"), row.names = FALSE)
  print(failure_table)
} else {
  message("Full nine-slide batch not started.")
}


========== slide1 ==========

Skipping complete result: slide1/EB_01

Skipping complete result: slide1/EB_02

Skipping complete result: slide1/EB_03

Skipping complete result: slide1/EB_04

Skipping complete result: slide1/EB_05

Skipping complete result: slide1/EB_06

Skipping complete result: slide1/EB_07

Skipping complete result: slide1/EB_08

Skipping complete result: slide1/EB_09

Skipping complete result: slide1/EB_10

Skipping complete result: slide1/EB_11

Skipping complete result: slide1/EB_12

Skipping complete result: slide1/EB_13

Skipping complete result: slide1/EB_14

Skipping complete result: slide1/EB_15

Skipping complete result: slide1/EB_16

Skipping complete result: slide1/EB_17

Skipping complete result: slide1/EB_18

Skipping complete result: slide1/EB_19

Skipping complete result: slide1/EB_20

Skipping complete result: slide1/EB_21

Skipping complete result: slide1/EB_22

Skipping complete result: slide1/EB_23

Skipping complete result: slide1/EB_24

Skipping 

[1] slide_EB error   
<0 rows> (or 0-length row.names)
